# Spectrogram Visualisation
Explores the log-mel features already extracted in `Development_data/feature/`.
No raw audio or GPU needed — run locally with the venv kernel.

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

from framework.metadata import SPECIES_NAMES, DOMAIN_NAMES

In [ ]:
# Load the training feature pickle
with open('Development_data/feature/training_features.pkl', 'rb') as f:
    train_payload = pickle.load(f)

with open('Development_data/feature/test_features.pkl', 'rb') as f:
    test_payload = pickle.load(f)

print(f"Training samples: {train_payload['num_items']}")
print(f"Test samples:     {test_payload['num_items']}")

In [ ]:
# Index samples by (species, domain) for easy lookup
def index_by_species_domain(payload):
    index = defaultdict(list)
    for item in payload['items']:
        index[(item['species'], item['domain'])].append(item)
    return index

train_index = index_by_species_domain(train_payload)
test_index  = index_by_species_domain(test_payload)

# Show which (species, domain) combinations exist in training
print('Training (species, domain) combinations:')
for key in sorted(train_index):
    print(f'  {key[0]:30s} {key[1]}  ({len(train_index[key])} clips)')

In [ ]:
def plot_spectrogram(ax, feature, title, max_frames=200):
    """Plot a single log-mel spectrogram. feature shape: [T, 64]"""
    display = feature[:max_frames].T  # [64, T]
    ax.imshow(display, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(title, fontsize=8)
    ax.set_xlabel('Time (frames)', fontsize=7)
    ax.set_ylabel('Mel bin', fontsize=7)
    ax.tick_params(labelsize=6)

In [ ]:
# --- Plot 1: One example spectrogram per species ---
# Uses the first available training sample for each species

fig, axes = plt.subplots(3, 3, figsize=(14, 9))
fig.suptitle('One example per species (training set)', fontsize=12)

for ax, species in zip(axes.flat, SPECIES_NAMES):
    # Find first domain that has this species
    sample = None
    for domain in DOMAIN_NAMES:
        items = train_index.get((species, domain), [])
        if items:
            sample = items[0]
            break
    if sample:
        plot_spectrogram(ax, sample['feature'], f"{species}\n({sample['domain']})")
    else:
        ax.set_visible(False)

plt.tight_layout()
plt.savefig('spectrogram_per_species.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to spectrogram_per_species.png')

In [ ]:
# --- Plot 2: Same species across all available domains ---
# Shows how much the spectrogram changes across recording conditions
# Change SPECIES below to explore others

SPECIES = 'Aedes aegypti'

available_domains = [
    (domain, train_index[(SPECIES, domain)])
    for domain in DOMAIN_NAMES
    if train_index[(SPECIES, domain)]
]

n = len(available_domains)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
if n == 1:
    axes = [axes]
fig.suptitle(f'{SPECIES} — same species, different domains', fontsize=12)

for ax, (domain, items) in zip(axes, available_domains):
    plot_spectrogram(ax, items[0]['feature'], f'Domain {domain}')

plt.tight_layout()
plt.savefig(f'spectrogram_domain_comparison_{SPECIES.replace(" ", "_")}.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')

In [ ]:
# --- Plot 3: Same domain, all species ---
# Shows what different species look like under the same recording conditions
# Change DOMAIN below to explore others

DOMAIN = 'D1'

available_species = [
    (species, train_index[(species, DOMAIN)])
    for species in SPECIES_NAMES
    if train_index[(species, DOMAIN)]
]

n = len(available_species)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
if n == 1:
    axes = [axes]
fig.suptitle(f'Domain {DOMAIN} — all species present', fontsize=12)

for ax, (species, items) in zip(axes, available_species):
    short_name = species.split()[-1]  # just the species epithet
    plot_spectrogram(ax, items[0]['feature'], short_name)

plt.tight_layout()
plt.savefig(f'spectrogram_species_in_{DOMAIN}.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved.')